In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 01 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 01.C1 — Compute a finite preimage

Given a function f, a finite candidate domain, and a target set, return every candidate x whose output lies in the target.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

An inverse image searches inputs. It does not require the function to have an inverse.

</details>

In [3]:
def preimage(f, domain, target):
    return {x for x in domain if f(x) in target}

In [4]:
assert preimage(lambda x:x*x,range(-3,4),{1,4})=={-2,-1,1,2}
frame=load_iris_data();mapping=dict(zip(frame.sample_id,frame.label))
ids=preimage(mapping.__getitem__,mapping,{0})
assert len(ids)==50
print("Recorded label-0 preimage contains",len(ids),"sample IDs")

Recorded label-0 preimage contains 50 sample IDs


**Why this works and what it does not establish:** For the square function two inputs can share an output. Both must be returned. The finite calculation illustrates the definition but does not enumerate an infinite real-line preimage.

## 01.C2 — Build equivalence classes

Partition the given integers by remainder modulo the positive integer m. Return a dictionary from remainders to sets.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The expression x % m gives the remainder. setdefault creates a group the first time it is needed.

</details>

In [5]:
def residue_classes(values, m):
    groups={}
    for x in values:
        groups.setdefault(x%m,set()).add(x)
    return groups

In [6]:
g=residue_classes(range(-3,4),3)
assert g[0]=={-3,0,3} and g[1]=={-2,1} and g[2]=={-1,2}
assert set().union(*g.values())==set(range(-3,4))
print("Classes:",g)

Classes: {0: {0, 3, -3}, 1: {1, -2}, 2: {2, -1}}


**Why this works and what it does not establish:** Every input belongs to exactly one class. This is a partition of the supplied integers. A quotient will later use classes themselves as new mathematical objects.

## 01.C3 — Multiply a matrix and vector by hand

For a rectangular numeric list A and a vector x of matching length, implement Ax with loops.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

For each output row multiply matching coordinates and add them.

</details>

In [7]:
def matvec(A, x):
    return [sum(a*b for a,b in zip(row,x)) for row in A]

In [8]:
assert matvec([[1,1],[0,0]],[2,-2])==[0,0]
A=load_iris_data().iloc[:3,1:5].to_numpy()
x=np.array([1,0,0,0])
assert np.allclose(matvec(A,x),A[:,0])
print("Selecting one measurement by matrix multiplication:",matvec(A,x))

Selecting one measurement by matrix multiplication: [np.float64(5.1), np.float64(4.9), np.float64(4.7)]


**Why this works and what it does not establish:** The assumptions require matching dimensions; zip alone is not a general input validator. Explain the map from a four-coordinate vector to a three-coordinate output for this particular matrix.

## 01.C4 — Connect rank to nullity

For an integer-valued matrix, return (rank, nullity) over F2 using the supplied rank_mod2. Explain why the number of columns appears.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

A has one input coordinate per column. The scalar field is part of the problem.

</details>

In [9]:
def rank_nullity_f2(A):
    A=np.asarray(A)
    rank=rank_mod2(A)
    return rank,A.shape[1]-rank

In [10]:
A=np.array([[1,1,0],[1,0,1],[0,1,1]])
assert rank_nullity_f2(A)==(2,1)
assert np.linalg.matrix_rank(A)==3
print("F2 rank/nullity",rank_nullity_f2(A),"versus real rank",np.linalg.matrix_rank(A))

F2 rank/nullity (2, 1) versus real rank 3


**Why this works and what it does not establish:** The same integer entries define different linear maps over different fields. The F2 sum of all three rows is zero, but the real-valued relation is different.

## 01.C5 — Center a measurement table

Return a copy of a finite nonempty feature matrix with each column mean subtracted. This exercise is exploratory, not a predictive preprocessing fit.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

axis=0 combines rows, leaving one value per feature.

</details>

In [11]:
def centered(X):
    X=np.asarray(X,dtype=float)
    return X-X.mean(axis=0)

In [12]:
X=load_iris_data().iloc[:,1:5].to_numpy()
Z=centered(X)
assert np.allclose(Z.mean(axis=0),0,atol=1e-12)
assert np.allclose(Z[0]-Z[1],X[0]-X[1])
print("Centered means:",Z.mean(axis=0))

Centered means: [-3.31586610e-16 -3.55271368e-16 -6.63173220e-16 -2.36847579e-16]


**Why this works and what it does not establish:** Subtracting the same vector from every point changes the coordinate origin but not pairwise differences. A learned predictive scaler, unlike this exploratory illustration, must fit only on the designated development partition.